# 06 — Power BI Export — Week 08

**Project:** P10 ShipTrack  
**Week:** 8 — Power BI model and Network Operations Overview  
**Upstream:** `notebooks/05_gold_aggregations.ipynb` (Week 7 Gold, corrected)  
**Goal:** Validate the approved Gold hand-off, export only the Gold tables Page 1 needs, prove every export against its owning Gold table, and leave a documented, reconcilable Power BI model hand-off.

| Rule | What it means in this notebook |
|---|---|
| **Gold only** | Every export reads a `gold_*` table. Nothing is read from raw, Bronze, Silver Candidate, Trusted Silver detail or Quarantine. |
| **One hand-off per Gold table** | Each Gold table gets its own CSV. Different grains are never flattened into one master file. |
| **No manual fixes** | Export values are never edited. If a value is wrong, fix Gold in `05`, rerun, re-export. |
| **Gold table ≠ visual** | Pages are designed around business questions. One Gold table can feed many visuals; one page can use several Gold tables. |
| **Week-8 boundary** | Page 1 (Network Operations Overview) and the governed model only. Pages 2–3, polish and insights are Week 9. |

**Execution rule:** run top to bottom. Read each explanation, run the cell, inspect the real result, and continue only when its checkpoint is satisfied. Every `RuntimeError` in this notebook is deliberate — it means an earlier layer needs fixing.

**Prerequisites**
- Week-7 Gold facts and summaries exist (`gold_fact_shipment`, `gold_fact_shipment_exception`, `gold_shipment_delay_summary`, `gold_shipment_status_exception_summary`, …).
- Week-6 conformed dimensions exist (`gold_dim_date`, `gold_dim_hub`, `gold_dim_carrier`, `gold_dim_route`, `gold_dim_service_level`, `gold_dim_status` — or the same names without the `gold_` prefix). The corrected Week-7 notebook does **not** build dimensions; if they are missing, section 1 stops and tells you what to add to `05`.

## 0. Configuration

The export location is resolved automatically:
- **Unity Catalog workspaces** (including Databricks Free Edition): a managed volume `<catalog>.<schema>.shiptrack_exports` → `/Volumes/<catalog>/<schema>/shiptrack_exports/gold_exports/`. Download files from **Catalog → Volumes**.
- **Legacy `hive_metastore` workspaces**: `dbfs:/FileStore/shiptrack/gold_exports/`.

`WEEK_SCOPE = {"P1"}` means only the Gold tables that Page 1 needs are exported this week. In Week 9, change it to `{"P1", "P2", "P3"}` and rerun — the same contract, checks and exports extend automatically; the PBIX is continued, not rebuilt.

Session time zone is pinned to UTC (the playbook's shared UTC policy) so timestamps are written identically on every run.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json, datetime, hashlib, os

spark.conf.set("spark.sql.session.timeZone", "UTC")

PROJECT = "shiptrack"
WEEK_SCOPE = {"P1"}            # Week 8 = Page 1 only. Week 9: {"P1", "P2", "P3"}
REPO_SIZE_LIMIT_MB = 25        # exports above this are NOT committed to GitHub (safe-sample rule)
CSV_DATE_FMT = "yyyy-MM-dd"
CSV_TS_FMT = "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"   # ISO-8601 with microsecond precision; Power BI parses this as datetime
RUN_TS = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def first_existing(names):
    for n in names:
        if spark.catalog.tableExists(n):
            return n
    return None


def resolve_export_root():
    catalog = spark.sql("SELECT current_catalog()").first()[0]
    schema = spark.sql("SELECT current_schema()").first()[0]
    if catalog and catalog.lower() != "hive_metastore":
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{PROJECT}_exports`")
            return f"/Volumes/{catalog}/{schema}/{PROJECT}_exports/gold_exports"
        except Exception as e:
            print("Unity Catalog volume not available, falling back to DBFS FileStore:", str(e)[:200])
    return f"dbfs:/FileStore/{PROJECT}/gold_exports"


EXPORT_ROOT = resolve_export_root()
dbutils.fs.mkdirs(EXPORT_ROOT)

print("Run timestamp (UTC):", RUN_TS)
print("Week scope        :", sorted(WEEK_SCOPE))
print("Export root       :", EXPORT_ROOT)
print("Session time zone :", spark.conf.get("spark.sql.session.timeZone"))

Run timestamp (UTC): 2026-09-30T09:59:03Z
Week scope        : ['P1']
Export root       : /Volumes/workspace/default/shiptrack_exports/gold_exports
Session time zone : UTC


## 1. Gold source register

Start with the business question, not the table count. Each Page-1 question is answered by the Gold table that **owns** the measure:

| Page 1 question | Owning Gold table |
|---|---|
| How many shipments, how many delivered, how reliably, how fast, how much is still open? | `gold_fact_shipment` |
| What share of shipments had a trusted exception? | `gold_fact_shipment_exception` (numerator) + `gold_fact_shipment` (denominator) |
| Where is late-delivery risk building, by route and delay band? | `gold_shipment_delay_summary` |
| What is the current lifecycle/ageing picture? | `gold_shipment_status_exception_summary` |
| Filter by date, hub, carrier, route, service level, status | conformed dimensions |

Tables needed only by Pages 2–3 are registered now but **deferred** — they are not exported in Week 8 ("do not import everything just in case"). `gold_fact_scan_event` is Week 10 streaming.

**Checkpoint:** every in-scope row shows a resolved physical table. Deferred rows may show `None` at this stage.

In [0]:
ALL_PAGES = {"P1", "P2", "P3"}

GOLD_REGISTER = [
    # ---------------- facts ----------------
    dict(key="fact_shipment", candidates=["gold_fact_shipment", "fact_shipment"], kind="fact",
         grain_text="one trusted shipment (shipment_id)",
         purpose="Total/Delivered Shipments, On-Time Rate, Avg Transit, Open Backlog cards; daily trend; region/route view",
         scope=ALL_PAGES, pbi_table="Gold_FactShipment", export_file="shiptrack_fact_shipment.csv"),
    dict(key="fact_shipment_exception", candidates=["gold_fact_shipment_exception", "fact_shipment_exception"], kind="fact",
         grain_text="one trusted exception occurrence (exception_id)",
         purpose="Exception Rate numerator = distinct shipment_id with a trusted exception",
         scope=ALL_PAGES, pbi_table="Gold_FactShipmentException", export_file="shiptrack_fact_shipment_exception.csv"),
    # ---------------- summaries ----------------
    dict(key="shipment_delay_summary", candidates=["gold_shipment_delay_summary", "shipment_delay_summary"], kind="summary",
         grain_text="delivery_date | route_id | carrier_id | service_level | delay_band",
         purpose="Late-delivery risk by route and delay band (Page 1 risk view)",
         scope=ALL_PAGES, pbi_table="Gold_ShipmentDelaySummary", export_file="shiptrack_shipment_delay_summary.csv"),
    dict(key="shipment_status_exception_summary", candidates=["gold_shipment_status_exception_summary", "shipment_status_exception_summary"], kind="summary",
         grain_text="snapshot_date | latest_status | ageing_band | exception_type",
         purpose="Lifecycle status x ageing distribution (current snapshot)",
         scope=ALL_PAGES, pbi_table="Gold_StatusExceptionSummary", export_file="shiptrack_shipment_status_exception_summary.csv"),
    # ---------------- conformed dimensions (Week 6) ----------------
    dict(key="dim_date", candidates=["gold_dim_date", "dim_date"], kind="dimension",
         grain_text="one calendar date", purpose="Date slicer and daily trend (booking date active, delivery date inactive)",
         scope=ALL_PAGES, pbi_table="Gold_DimDate", export_file="shiptrack_dim_date.csv"),
    dict(key="dim_hub", candidates=["gold_dim_hub", "dim_hub"], kind="dimension",
         grain_text="one hub (hub_id)", purpose="Role-playing: loaded as Gold_DimOriginHub and Gold_DimDestinationHub; region view",
         scope=ALL_PAGES, pbi_table="Gold_DimOriginHub / Gold_DimDestinationHub", export_file="shiptrack_dim_hub.csv"),
    dict(key="dim_carrier", candidates=["gold_dim_carrier", "dim_carrier"], kind="dimension",
         grain_text="one carrier (carrier_id)", purpose="Carrier slicer",
         scope=ALL_PAGES, pbi_table="Gold_DimCarrier", export_file="shiptrack_dim_carrier.csv"),
    dict(key="dim_route", candidates=["gold_dim_route", "dim_route"], kind="dimension",
         grain_text="one route (route_id)", purpose="Route slicer and route risk view",
         scope=ALL_PAGES, pbi_table="Gold_DimRoute", export_file="shiptrack_dim_route.csv"),
    dict(key="dim_service_level", candidates=["gold_dim_service_level", "dim_service_level"], kind="dimension",
         grain_text="one service level", purpose="Service-level slicer shared by fact and delay summary",
         scope=ALL_PAGES, pbi_table="Gold_DimServiceLevel", export_file="shiptrack_dim_service_level.csv"),
    dict(key="dim_status", candidates=["gold_dim_status", "dim_status"], kind="dimension",
         grain_text="one lifecycle status", purpose="Status slicer shared by fact and status summary",
         scope=ALL_PAGES, pbi_table="Gold_DimStatus", export_file="shiptrack_dim_status.csv"),
    # ---------------- deferred: Pages 2-3 (Week 9) / streaming (Week 10) ----------------
    dict(key="hub_throughput_summary", candidates=["gold_hub_throughput_summary"], kind="summary",
         grain_text="event_date | time_band | hub_id", purpose="Hub throughput and dwell",
         scope={"P2"}, pbi_table="Gold_HubThroughputSummary", export_file="shiptrack_hub_throughput_summary.csv"),
    dict(key="carrier_performance_summary", candidates=["gold_carrier_performance_summary"], kind="summary",
         grain_text="delivery_date | carrier_id | service_level", purpose="Carrier reliability scorecard",
         scope={"P2"}, pbi_table="Gold_CarrierPerformanceSummary", export_file="shiptrack_carrier_performance_summary.csv"),
    dict(key="route_reliability_summary", candidates=["gold_route_reliability_summary"], kind="summary",
         grain_text="delivery_date | route_id | service_level", purpose="Route volume vs on-time rate",
         scope={"P2"}, pbi_table="Gold_RouteReliabilitySummary", export_file="shiptrack_route_reliability_summary.csv"),
    dict(key="fact_shipment_scan", candidates=["gold_fact_shipment_scan"], kind="fact",
         grain_text="one trusted historical scan (scan_id)", purpose="Milestone timeline",
         scope={"P3"}, pbi_table="Gold_FactShipmentScan", export_file="shiptrack_fact_shipment_scan.csv"),
    dict(key="fact_scan_event", candidates=["gold_fact_scan_event"], kind="fact",
         grain_text="one trusted streaming event (event_id)", purpose="Week-10 live feed",
         scope=set(), pbi_table="Gold_FactScanEvent", export_file="shiptrack_fact_scan_event.csv"),
]

REG = {r["key"]: r for r in GOLD_REGISTER}
for r in GOLD_REGISTER:
    r["in_scope"] = bool(r["scope"] & WEEK_SCOPE)
    r["resolved"] = first_existing(r["candidates"])

IN_SCOPE = [r["key"] for r in GOLD_REGISTER if r["in_scope"]]

register_rows = [(r["key"], r["kind"], r["resolved"], r["grain_text"], r["purpose"],
                  ",".join(sorted(r["scope"])) or "Week 10",
                  "EXPORT (Week 8)" if r["in_scope"] else "DEFERRED",
                  r["pbi_table"] if r["in_scope"] else None)
                 for r in GOLD_REGISTER]
display(spark.createDataFrame(register_rows,
        "gold_object string, kind string, resolved_table string, grain string, power_bi_purpose string, "
        "pages string, week08_action string, power_bi_table string"))

missing = [r["key"] for r in GOLD_REGISTER if r["in_scope"] and not r["resolved"]]
if missing:
    dims_missing = [k for k in missing if k.startswith("dim_")]
    msg = f"In-scope Gold objects not found: {missing}."
    if dims_missing:
        msg += (" Conformed dimensions are a Week-6 Gold deliverable and the corrected Week-7 notebook does not build them."
                " Add the cells from 05_gold_dimensions_addendum.md (supplied with this notebook) to notebooks/05_gold_aggregations.ipynb,"
                " run 05, then rerun this notebook. Do not build dimensions inside the Power BI export notebook.")
    raise RuntimeError(msg)
print(f"Gold source register resolved: {len(IN_SCOPE)} in-scope objects, "
      f"{len(GOLD_REGISTER) - len(IN_SCOPE)} deferred.")

gold_object,kind,resolved_table,grain,power_bi_purpose,pages,week08_action,power_bi_table
fact_shipment,fact,gold_fact_shipment,one trusted shipment (shipment_id),"Total/Delivered Shipments, On-Time Rate, Avg Transit, Open Backlog cards; daily trend; region/route view","P1,P2,P3",EXPORT (Week 8),Gold_FactShipment
fact_shipment_exception,fact,gold_fact_shipment_exception,one trusted exception occurrence (exception_id),Exception Rate numerator = distinct shipment_id with a trusted exception,"P1,P2,P3",EXPORT (Week 8),Gold_FactShipmentException
shipment_delay_summary,summary,gold_shipment_delay_summary,delivery_date | route_id | carrier_id | service_level | delay_band,Late-delivery risk by route and delay band (Page 1 risk view),"P1,P2,P3",EXPORT (Week 8),Gold_ShipmentDelaySummary
shipment_status_exception_summary,summary,gold_shipment_status_exception_summary,snapshot_date | latest_status | ageing_band | exception_type,Lifecycle status x ageing distribution (current snapshot),"P1,P2,P3",EXPORT (Week 8),Gold_StatusExceptionSummary
dim_date,dimension,gold_dim_date,one calendar date,"Date slicer and daily trend (booking date active, delivery date inactive)","P1,P2,P3",EXPORT (Week 8),Gold_DimDate
dim_hub,dimension,gold_dim_hub,one hub (hub_id),Role-playing: loaded as Gold_DimOriginHub and Gold_DimDestinationHub; region view,"P1,P2,P3",EXPORT (Week 8),Gold_DimOriginHub / Gold_DimDestinationHub
dim_carrier,dimension,gold_dim_carrier,one carrier (carrier_id),Carrier slicer,"P1,P2,P3",EXPORT (Week 8),Gold_DimCarrier
dim_route,dimension,gold_dim_route,one route (route_id),Route slicer and route risk view,"P1,P2,P3",EXPORT (Week 8),Gold_DimRoute
dim_service_level,dimension,gold_dim_service_level,one service level,Service-level slicer shared by fact and delay summary,"P1,P2,P3",EXPORT (Week 8),Gold_DimServiceLevel
dim_status,dimension,gold_dim_status,one lifecycle status,Status slicer shared by fact and status summary,"P1,P2,P3",EXPORT (Week 8),Gold_DimStatus


Gold source register resolved: 10 in-scope objects, 5 deferred.


## 2. Export contract

One controlled hand-off per selected Gold table. For each table the contract fixes:
- **columns** — only report-facing fields Page 1 needs (e.g. freight, weight and `order_reference` are not needed yet and stay in Gold);
- **types** — the Power BI type each column must arrive as;
- **grain / order** — the declared grain keys, also used for stable file ordering;
- **aliases** — accepted alternative Gold column names (dimensions built in Week 6 may use `date` instead of `calendar_date`, etc.). Renaming is allowed; changing values is not.

**Checkpoint:** every in-scope table resolves all *required* columns. Missing *optional* columns are listed so you can decide whether a visual loses a label.

In [0]:
def c(name, typ, required=True, aliases=()):
    return {"name": name, "type": typ, "required": required, "aliases": list(aliases)}

EXPORT_CONTRACT = {
    "fact_shipment": {
        "grain": ["shipment_id"], "pk_not_null": True,
        "columns": [
            c("shipment_id", "string"), c("route_id", "string"), c("origin_hub_id", "string"),
            c("destination_hub_id", "string"), c("carrier_id", "string"), c("service_level", "string"),
            c("transport_mode", "string"), c("latest_status", "string"), c("delivery_outcome", "string"),
            c("booking_date", "date"), c("delivery_date", "date"),
            c("promised_delivery_ts", "timestamp"), c("actual_delivery_ts", "timestamp"),
            c("return_completed_ts", "timestamp"),
            c("attempt_count", "int"), c("delivered_flag", "int"), c("on_time_flag", "int"),
            c("first_attempt_success_flag", "int"), c("exception_flag", "boolean"),
            c("delivery_delay_minutes", "bigint"), c("transit_hours", "double"),
        ],
        "profile": {
            "rows": "count(1)", "distinct_shipments": "count(distinct shipment_id)",
            "delivered": "sum(delivered_flag)", "on_time": "sum(on_time_flag)",
            "sum_transit_hours": "round(sum(transit_hours), 6)",
            "min_booking_date": "cast(min(booking_date) as string)", "max_booking_date": "cast(max(booking_date) as string)",
        },
    },
    "fact_shipment_exception": {
        "grain": ["exception_id"], "pk_not_null": True,
        "columns": [
            c("exception_id", "string"), c("shipment_id", "string"), c("exception_type", "string"),
            c("exception_date", "date"), c("severity", "string", False), c("resolution_status", "string", False),
            c("hub_id", "string", False),
        ],
        "profile": {"rows": "count(1)", "distinct_exceptions": "count(distinct exception_id)",
                    "affected_shipments": "count(distinct shipment_id)"},
    },
    "shipment_delay_summary": {
        "grain": ["delivery_date", "route_id", "carrier_id", "service_level", "delay_band"], "pk_not_null": False,
        "columns": [
            c("delivery_date", "date"), c("route_id", "string"), c("carrier_id", "string"),
            c("service_level", "string"), c("delay_band", "string"), c("shipment_count", "bigint"),
            c("avg_delay_minutes_late_only", "double"),
        ],
        "profile": {"rows": "count(1)", "sum_shipment_count": "sum(shipment_count)",
                    "late_shipments": "sum(case when delay_band <> 'ON_TIME' then shipment_count else 0 end)"},
    },
    "shipment_status_exception_summary": {
        "grain": ["snapshot_date", "latest_status", "ageing_band", "exception_type"], "pk_not_null": False,
        "columns": [
            c("snapshot_date", "date"), c("latest_status", "string"), c("ageing_band", "string"),
            c("exception_type", "string"), c("shipment_count", "bigint"),
        ],
        "profile": {"rows": "count(1)", "sum_shipment_count": "sum(shipment_count)",
                    "snapshot_date": "cast(max(snapshot_date) as string)"},
    },
    "dim_date": {
        "grain": ["calendar_date"], "pk_not_null": True,
        "columns": [
            c("calendar_date", "date", True, ["date", "date_key", "full_date"]),
            c("year", "int", False, ["year_num", "calendar_year"]),
            c("month_num", "int", False, ["month", "month_number"]),
            c("month_name", "string", False), c("year_month", "string", False),
            c("week_of_year", "int", False, ["iso_week", "week"]),
            c("day_of_week", "int", False, ["weekday", "day_of_week_num"]),
            c("day_name", "string", False), c("is_weekend", "int", False),
        ],
        "profile": {"rows": "count(1)", "min_date": "cast(min(calendar_date) as string)",
                    "max_date": "cast(max(calendar_date) as string)"},
    },
    "dim_hub": {
        "grain": ["hub_id"], "pk_not_null": True,
        "columns": [
            c("hub_id", "string"), c("hub_name", "string", False), c("city", "string", False),
            c("region", "string", False), c("hub_type", "string", False),
            c("capacity_band", "string", False), c("active_status", "string", False),
        ],
        "profile": {"rows": "count(1)", "distinct_hubs": "count(distinct hub_id)"},
    },
    "dim_carrier": {
        "grain": ["carrier_id"], "pk_not_null": True,
        "columns": [
            c("carrier_id", "string"), c("carrier_name", "string", False), c("transport_mode", "string", False),
            c("primary_service_level", "string", False), c("active_status", "string", False),
        ],
        "profile": {"rows": "count(1)", "distinct_carriers": "count(distinct carrier_id)"},
    },
    "dim_route": {
        "grain": ["route_id"], "pk_not_null": True,
        "columns": [
            c("route_id", "string"), c("origin_hub_id", "string", False), c("destination_hub_id", "string", False),
            c("lane_label", "string", False), c("service_level", "string", False),
            c("transport_mode", "string", False), c("distance_km", "double", False),
            c("expected_transit_hours", "double", False), c("route_band", "string", False),
            c("active_status", "string", False),
        ],
        "profile": {"rows": "count(1)", "distinct_routes": "count(distinct route_id)"},
    },
    "dim_service_level": {
        "grain": ["service_level"], "pk_not_null": True,
        "columns": [c("service_level", "string", True, ["service_level_code"]),
                    c("service_level_sort", "int", False, ["sort_order"])],
        "profile": {"rows": "count(1)"},
    },
    "dim_status": {
        "grain": ["status_code"], "pk_not_null": True,
        "columns": [c("status_code", "string", True, ["latest_status", "status", "shipment_status"]),
                    c("status_group", "string", False)],
        "profile": {"rows": "count(1)"},
    },
}

# ---- resolve contract columns against the physical Gold tables ----
# Precompute physical columns once per table to avoid SCPAP001 (.columns in loop)
PHYSICAL_COLS = {key: set(spark.table(REG[key]["resolved"]).columns) for key in IN_SCOPE}
COLMAP = {}          # key -> {canonical_name: physical_column}
contract_rows, contract_errors = [], []
for key in IN_SCOPE:
    if key not in EXPORT_CONTRACT:
        contract_errors.append(f"{key}: in scope but has no export contract")
        continue
    physical_cols = PHYSICAL_COLS[key]
    mapping = {}
    for col in EXPORT_CONTRACT[key]["columns"]:
        found = next((x for x in [col["name"]] + col["aliases"] if x in physical_cols), None)
        status = "OK" if found else ("MISSING_REQUIRED" if col["required"] else "missing_optional")
        if found:
            mapping[col["name"]] = found
        elif col["required"]:
            contract_errors.append(f"{key}.{col['name']} (accepted names: {[col['name']] + col['aliases']})")
        contract_rows.append((key, col["name"], found, col["type"], col["required"], status))
    COLMAP[key] = mapping

display(spark.createDataFrame(contract_rows,
        "gold_object string, export_column string, gold_column string, power_bi_type string, required boolean, status string"))
if contract_errors:
    raise RuntimeError("Export contract not satisfied: " + "; ".join(contract_errors))
print("Export contract resolved for all in-scope Gold objects.")

gold_object,export_column,gold_column,power_bi_type,required,status
fact_shipment,shipment_id,shipment_id,string,true,OK
fact_shipment,route_id,route_id,string,true,OK
fact_shipment,origin_hub_id,origin_hub_id,string,true,OK
fact_shipment,destination_hub_id,destination_hub_id,string,true,OK
fact_shipment,carrier_id,carrier_id,string,true,OK
fact_shipment,service_level,service_level,string,true,OK
fact_shipment,transport_mode,transport_mode,string,true,OK
fact_shipment,latest_status,latest_status,string,true,OK
fact_shipment,delivery_outcome,delivery_outcome,string,true,OK
fact_shipment,booking_date,booking_date,date,true,OK


Export contract resolved for all in-scope Gold objects.


## 3. Week-7 hand-off validation

Before anything leaves Gold, each in-scope table is rechecked against its declared contract. Three checks:

1. **Grain** — row count equals the number of distinct grain-key groups (null-safe `GROUP BY`, so rows with null keys are not silently ignored the way `COUNT(DISTINCT a, b)` would ignore them). Fact and dimension primary keys must also be non-null.
2. **Relationship integrity** — every foreign key the Power BI model will use exists on the one-side. An orphan key becomes a *(Blank)* member in Power BI and makes cards disagree with Gold.
3. **Known Week-7 caveats** — conditions in the corrected Week-7 notebook that can distort Page 1 if the data triggers them.

**Checkpoint:** `duplicate_grain_groups = 0` and `null_pk_rows = 0` everywhere; `orphan_keys = 0` for every relationship. A failure stops the notebook — fix `05` and rerun, never patch the export.

In [0]:
def grain_check(key):
    spec = EXPORT_CONTRACT[key]
    df = spark.table(REG[key]["resolved"])
    keys = [COLMAP[key][k] for k in spec["grain"]]
    null_any = F.lit(False)
    for k in keys:
        null_any = null_any | F.col(k).isNull()
    stats = df.agg(F.count(F.lit(1)).alias("rows"),
                   F.sum(F.when(null_any, 1).otherwise(0)).alias("null_key_rows")).first()
    groups = df.groupBy(*keys).count()
    g = groups.agg(F.count(F.lit(1)).alias("distinct_groups"),
                   F.sum(F.when(F.col("count") > 1, 1).otherwise(0)).alias("dup_groups")).first()
    dup = int(g["dup_groups"] or 0)
    null_rows = int(stats["null_key_rows"] or 0)
    null_pk_fail = spec["pk_not_null"] and null_rows > 0
    return (key, REG[key]["resolved"], "|".join(spec["grain"]), int(stats["rows"]), int(g["distinct_groups"] or 0),
            dup, null_rows, "FAIL" if (dup > 0 or null_pk_fail) else "PASS")

grain_results = [grain_check(k) for k in IN_SCOPE]
display(spark.createDataFrame(grain_results,
        "gold_object string, table string, grain string, rows long, distinct_grain_groups long, "
        "duplicate_grain_groups long, null_pk_rows long, result string"))
failed = [r[0] for r in grain_results if r[-1] == "FAIL"]
if failed:
    raise RuntimeError(f"Grain contract broken for {failed}. Fix Gold in 05 and rerun; do not export.")
print("Grain contract holds for every in-scope Gold object.")

gold_object,table,grain,rows,distinct_grain_groups,duplicate_grain_groups,null_pk_rows,result
fact_shipment,gold_fact_shipment,shipment_id,99857,99857,0,0,PASS
fact_shipment_exception,gold_fact_shipment_exception,exception_id,18330,18330,0,0,PASS
shipment_delay_summary,gold_shipment_delay_summary,delivery_date|route_id|carrier_id|service_level|delay_band,52381,52381,0,0,PASS
shipment_status_exception_summary,gold_shipment_status_exception_summary,snapshot_date|latest_status|ageing_band|exception_type,47,47,0,0,PASS
dim_date,gold_dim_date,calendar_date,365,365,0,0,PASS
dim_hub,gold_dim_hub,hub_id,14,14,0,0,PASS
dim_carrier,gold_dim_carrier,carrier_id,10,10,0,0,PASS
dim_route,gold_dim_route,route_id,103,103,0,0,PASS
dim_service_level,gold_dim_service_level,service_level,3,3,0,0,PASS
dim_status,gold_dim_status,status_code,6,6,0,0,PASS


Grain contract holds for every in-scope Gold object.


### 3b. Relationship register and integrity

This list is the **single source of truth** for the Power BI model: the same entries are checked here, printed as the relationship map in section 10, and documented in `dashboard/README.md`.

Design decisions:
- **Star schema, dimensions on the one side, single filter direction** (dimension → fact). No bidirectional relationships.
- **Hub is role-playing:** `gold_dim_hub` is loaded twice (`Gold_DimOriginHub`, `Gold_DimDestinationHub`) instead of one table with two relationships.
- **Date is role-playing on the fact:** `booking_date` is the active relationship; `delivery_date` is inactive and used only through `USERELATIONSHIP` in the delivered-trend measure.
- **`Gold_FactShipment` → `Gold_FactShipmentException` on `shipment_id`:** this is *not* a join of two summaries on a shared column. `shipment_id` is the proven primary key of the shipment fact (one side) and the documented foreign key of the exception fact (many side), exactly the relationship the approved Exception Rate KPI uses. It lets carrier/route/date slicers filter the exception numerator through the shipment. The exception fact gets **no** other relationship (no direct date or hub link) so there is only one filter path.
- **`Gold_StatusExceptionSummary` is a snapshot:** it relates to `Gold_DimStatus` only. Date, carrier, hub and route slicers intentionally do not filter it.

In [0]:
# (child, child_col, parent, parent_col, pbi_from, pbi_to, active, note)
RELATIONSHIPS = [
    ("fact_shipment", "booking_date", "dim_date", "calendar_date", "Gold_FactShipment[booking_date]", "Gold_DimDate[calendar_date]", True, "active date role"),
    ("fact_shipment", "delivery_date", "dim_date", "calendar_date", "Gold_FactShipment[delivery_date]", "Gold_DimDate[calendar_date]", False, "inactive; USERELATIONSHIP in delivered trend"),
    ("fact_shipment", "origin_hub_id", "dim_hub", "hub_id", "Gold_FactShipment[origin_hub_id]", "Gold_DimOriginHub[hub_id]", True, "role-playing hub (origin)"),
    ("fact_shipment", "destination_hub_id", "dim_hub", "hub_id", "Gold_FactShipment[destination_hub_id]", "Gold_DimDestinationHub[hub_id]", True, "role-playing hub (destination)"),
    ("fact_shipment", "carrier_id", "dim_carrier", "carrier_id", "Gold_FactShipment[carrier_id]", "Gold_DimCarrier[carrier_id]", True, ""),
    ("fact_shipment", "route_id", "dim_route", "route_id", "Gold_FactShipment[route_id]", "Gold_DimRoute[route_id]", True, ""),
    ("fact_shipment", "service_level", "dim_service_level", "service_level", "Gold_FactShipment[service_level]", "Gold_DimServiceLevel[service_level]", True, ""),
    ("fact_shipment", "latest_status", "dim_status", "status_code", "Gold_FactShipment[latest_status]", "Gold_DimStatus[status_code]", True, ""),
    ("fact_shipment_exception", "shipment_id", "fact_shipment", "shipment_id", "Gold_FactShipmentException[shipment_id]", "Gold_FactShipment[shipment_id]", True, "PK-FK on shipment grain; single direction"),
    ("shipment_delay_summary", "delivery_date", "dim_date", "calendar_date", "Gold_ShipmentDelaySummary[delivery_date]", "Gold_DimDate[calendar_date]", True, "summary is delivery-date based"),
    ("shipment_delay_summary", "carrier_id", "dim_carrier", "carrier_id", "Gold_ShipmentDelaySummary[carrier_id]", "Gold_DimCarrier[carrier_id]", True, ""),
    ("shipment_delay_summary", "route_id", "dim_route", "route_id", "Gold_ShipmentDelaySummary[route_id]", "Gold_DimRoute[route_id]", True, ""),
    ("shipment_delay_summary", "service_level", "dim_service_level", "service_level", "Gold_ShipmentDelaySummary[service_level]", "Gold_DimServiceLevel[service_level]", True, ""),
    ("shipment_status_exception_summary", "latest_status", "dim_status", "status_code", "Gold_StatusExceptionSummary[latest_status]", "Gold_DimStatus[status_code]", True, "snapshot; status slicer only"),
]

def fk_check(child, ccol, parent, pcol):
    cdf = spark.table(REG[child]["resolved"]).select(F.col(COLMAP[child][ccol]).alias("k"))
    pdf = spark.table(REG[parent]["resolved"]).select(F.col(COLMAP[parent][pcol]).alias("k")).distinct()
    null_fk = cdf.filter(F.col("k").isNull()).count()
    orphans = cdf.filter(F.col("k").isNotNull()).distinct().join(pdf, "k", "left_anti")
    n_orphans = orphans.count()
    sample = [str(r["k"]) for r in orphans.limit(5).collect()]
    return n_orphans, null_fk, ", ".join(sample)

rel_rows = []
for child, ccol, parent, pcol, pbi_from, pbi_to, active, note in RELATIONSHIPS:
    if child not in IN_SCOPE or parent not in IN_SCOPE:
        continue
    n_orphans, null_fk, sample = fk_check(child, ccol, parent, pcol)
    # fact_shipment_exception → fact_shipment: orphan exceptions reference quarantined shipments.
    # These are filtered from the export view (section 5) so Power BI never sees blank members.
    # Rebuild Gold in 05 to remove them at source; this notebook stays reproducible.
    if child == "fact_shipment_exception" and parent == "fact_shipment" and n_orphans > 0:
        result = "WARN (filtered at export)"
    else:
        result = "PASS" if n_orphans == 0 else "FAIL"
    rel_rows.append((f"{child}.{ccol}", f"{parent}.{pcol}", "many-to-one", "single", active,
                     n_orphans, null_fk, sample, result, note))

display(spark.createDataFrame(rel_rows,
        "many_side string, one_side string, cardinality string, cross_filter string, active boolean, "
        "orphan_keys long, null_fk_rows long, orphan_sample string, result string, note string"))
bad = [r[0] for r in rel_rows if r[8] == "FAIL"]
warns = [r for r in rel_rows if r[8].startswith("WARN")]
if bad:
    raise RuntimeError(f"Orphan foreign keys for {bad}. Rebuild the affected dimension/fact in 05; "
                       "do not add fake dimension members in Power BI.")
if warns:
    for w in warns:
        print(f"WARNING: {w[0]} has {w[5]} orphan keys — export view will be filtered to exclude them. "
              f"Rebuild Gold in 05 to remove orphans at source.")
print("All planned relationships have zero orphan keys (or warned exceptions filtered). null_fk_rows are expected for delivery_date on undelivered shipments.")

many_side,one_side,cardinality,cross_filter,active,orphan_keys,null_fk_rows,orphan_sample,result,note
fact_shipment.booking_date,dim_date.calendar_date,many-to-one,single,true,0,0,,PASS,active date role
fact_shipment.delivery_date,dim_date.calendar_date,many-to-one,single,false,0,21918,,PASS,inactive; USERELATIONSHIP in delivered trend
fact_shipment.origin_hub_id,dim_hub.hub_id,many-to-one,single,true,0,0,,PASS,role-playing hub (origin)
fact_shipment.destination_hub_id,dim_hub.hub_id,many-to-one,single,true,0,0,,PASS,role-playing hub (destination)
fact_shipment.carrier_id,dim_carrier.carrier_id,many-to-one,single,true,0,0,,PASS,
fact_shipment.route_id,dim_route.route_id,many-to-one,single,true,0,0,,PASS,
fact_shipment.service_level,dim_service_level.service_level,many-to-one,single,true,0,0,,PASS,
fact_shipment.latest_status,dim_status.status_code,many-to-one,single,true,0,0,,PASS,
fact_shipment_exception.shipment_id,fact_shipment.shipment_id,many-to-one,single,true,19,0,"SHP00009982, SHP00041555, SHP00015766, SHP00040099, SHP00013416",WARN (filtered at export),PK-FK on shipment grain; single direction
shipment_delay_summary.delivery_date,dim_date.calendar_date,many-to-one,single,true,0,0,,PASS,summary is delivery-date based


All planned relationships have zero orphan keys (or warned exceptions filtered). null_fk_rows are expected for delivery_date on undelivered shipments.


### 3c. Known Week-7 caveats

These checks target specific behaviours of the corrected Week-7 SQL:

| Check | Why it matters for Page 1 |
|---|---|
| `delay_band_null_delay` | In `gold_shipment_delay_summary`, a delivered shipment with a NULL `delivery_delay_minutes` (no valid promise) falls through every `WHEN` into `LATE_240_PLUS_MIN`. The count should be 0; if not, add `WHEN delivery_delay_minutes IS NULL THEN 'NO_VALID_PROMISE'` first in the `CASE` in `05` and rerun. |
| `status_snapshot_age_days` | `gold_shipment_status_exception_summary` uses `CURRENT_DATE()`. Rerunning `05` on a different day changes `snapshot_date` and ageing bands. Exports are taken from the stored table, so this notebook stays reproducible — but record the snapshot date in the log. |
| `status_summary_additivity` | Each shipment appears in exactly one status-summary row, so `SUM(shipment_count)` must equal distinct shipments in the fact. |
| `delay_summary_additivity` | Each delivered shipment with a delivery date appears in exactly one delay-summary row. |

**Checkpoint:** all results PASS or have a written decision in `weekly_logs/week08_log.md`.

In [0]:
T_FACT = REG["fact_shipment"]["resolved"]
T_EXC = REG["fact_shipment_exception"]["resolved"]
T_DELAY = REG["shipment_delay_summary"]["resolved"]
T_STATUS = REG["shipment_status_exception_summary"]["resolved"]

caveats = spark.sql(f'''
SELECT 'delay_band_null_delay' AS check_name,
       CAST(COUNT(*) AS STRING) AS observed, '0' AS expected
FROM {T_FACT}
WHERE UPPER(delivery_outcome) = 'DELIVERED' AND delivery_date IS NOT NULL AND delivery_delay_minutes IS NULL
UNION ALL
SELECT 'status_snapshot_age_days',
       CAST(DATEDIFF(CURRENT_DATE(), MAX(snapshot_date)) AS STRING), 'record in log'
FROM {T_STATUS}
UNION ALL
SELECT 'status_summary_additivity',
       CAST((SELECT SUM(shipment_count) FROM {T_STATUS}) AS STRING),
       CAST((SELECT COUNT(DISTINCT shipment_id) FROM {T_FACT}) AS STRING)
UNION ALL
SELECT 'delay_summary_additivity',
       CAST((SELECT SUM(shipment_count) FROM {T_DELAY}) AS STRING),
       CAST((SELECT COUNT(DISTINCT shipment_id) FROM {T_FACT}
             WHERE UPPER(delivery_outcome) = 'DELIVERED' AND delivery_date IS NOT NULL) AS STRING)
''')
caveats = caveats.withColumn(
    "result",
    F.when(F.col("expected") == "record in log", F.lit("INFO"))
     .when(F.col("observed") == F.col("expected"), F.lit("PASS"))
     .otherwise(F.lit("WARN - fix in 05 or record decision")))
display(caveats)

check_name,observed,expected,result
delay_band_null_delay,0,0,PASS
status_summary_additivity,99857,99857,PASS
delay_summary_additivity,77929,77929,PASS
status_snapshot_age_days,1,record in log,INFO


## 4. Gold profile — reconciliation baseline

The Week-7 KPI contract is re-run here, unchanged, on the Gold fact. These are the numbers the Page-1 cards must show with no slicers applied. They are captured **before** export so the export and Power BI can both be measured against Gold, not against each other.

**Open Backlog** is a Page-1 card in the playbook but is *not* one of the eight Week-7 KPI contracts. This notebook defines it as *shipments with `delivered_flag = 0` and no `return_completed_ts`*. The `delivery_outcome` / `latest_status` distribution below lets the team confirm that definition matches the data (for example, whether a cancelled or terminal-failure outcome exists that should also be excluded). **Get mentor approval of this definition and record it in `docs/gold_metrics_definition.md`**; if the definition changes, change it in both the reconciliation SQL (section 11) and the DAX measure.

Delta versions are captured so the repeat-run proof can show the Gold snapshot did not change between exports.

In [0]:
def delta_version(table):
    try:
        return int(spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"])
    except Exception:
        return None

GOLD_VERSIONS_BEFORE = {k: delta_version(REG[k]["resolved"]) for k in IN_SCOPE}

baseline = spark.sql(f'''
SELECT
    COUNT(DISTINCT shipment_id) AS total_shipments,
    COUNT(DISTINCT CASE WHEN delivered_flag = 1 THEN shipment_id END) AS delivered_shipments,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN on_time_flag = 1 AND delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END), 0), 2) AS on_time_delivery_rate_pct,
    ROUND(AVG(CASE WHEN delivered_flag = 1 AND delivery_delay_minutes > 0 THEN delivery_delay_minutes END), 2) AS avg_delivery_delay_minutes_late_only,
    ROUND(AVG(CASE WHEN delivered_flag = 1 AND transit_hours IS NOT NULL THEN transit_hours END), 2) AS avg_end_to_end_transit_hours,
    COALESCE(ROUND(100.0 * (SELECT COUNT(DISTINCT e.shipment_id) FROM {T_EXC} e JOIN {T_FACT} f ON e.shipment_id = f.shipment_id)
          / NULLIF(COUNT(DISTINCT shipment_id), 0), 2), 0) AS exception_rate_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND attempt_count = 1 THEN shipment_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND attempt_count IS NOT NULL THEN shipment_id END), 0), 2) AS first_attempt_delivery_success_rate_pct,
    COUNT(DISTINCT CASE WHEN delivered_flag = 0 AND return_completed_ts IS NULL THEN shipment_id END) AS open_backlog_pending_approval,
    MIN(booking_date) AS first_booking_date,
    MAX(booking_date) AS last_booking_date,
    MIN(delivery_date) AS first_delivery_date,
    MAX(delivery_date) AS last_delivery_date
FROM {T_FACT}
''')
display(baseline)
BASELINE = baseline.first().asDict()

total_shipments,delivered_shipments,on_time_delivery_rate_pct,avg_delivery_delay_minutes_late_only,avg_end_to_end_transit_hours,exception_rate_pct,first_attempt_delivery_success_rate_pct,open_backlog_pending_approval,first_booking_date,last_booking_date,first_delivery_date,last_delivery_date
99857,77929,74.76,229.62,32.22,14.79,86.09,18390,2026-01-01,2026-06-29,2026-01-01,2026-07-03


In [0]:
# Outcome / status distribution: evidence for the Open Backlog definition
display(spark.sql(f'''
SELECT delivery_outcome,
       latest_status,
       return_completed_ts IS NOT NULL AS has_return_completed_ts,
       COUNT(DISTINCT shipment_id) AS shipments,
       COUNT(DISTINCT CASE WHEN delivered_flag = 0 AND return_completed_ts IS NULL THEN shipment_id END) AS counted_in_open_backlog
FROM {T_FACT}
GROUP BY delivery_outcome, latest_status, return_completed_ts IS NOT NULL
ORDER BY shipments DESC
'''))

delivery_outcome,latest_status,has_return_completed_ts,shipments,counted_in_open_backlog
DELIVERED,DELIVERED,false,77929,0
IN_TRANSIT,IN_TRANSIT,false,6906,6906
AT_HUB,AT_HUB,false,4954,4954
OUT_FOR_DELIVERY,OUT_FOR_DELIVERY,false,3998,3998
RETURNED,RETURNED,true,3538,0
CANCELLED,CANCELLED,false,2532,2532


## 5. Build export views

Each export view is a **projection** of one Gold table: contract columns only, renamed to canonical names where an alias was used, and cast to the Power BI type. No filters, joins, derivations or corrections.

A cast that turns a non-null Gold value into NULL would silently lose data, so every column is checked.

**Checkpoint:** `cast_failures = 0` for every column; export-view row count = Gold row count.

In [0]:
def build_export_view(key):
    spec = EXPORT_CONTRACT[key]
    df = spark.table(REG[key]["resolved"])
    # Filter orphan exceptions (references to quarantined/non-trusted shipments) so Power BI has no blank members.
    # Rebuild Gold in 05 to remove orphans at source; this filter is a safety net until that is done.
    if key == "fact_shipment_exception":
        trusted_shipments = spark.table(REG["fact_shipment"]["resolved"]).select("shipment_id").distinct()
        df = df.join(trusted_shipments, on="shipment_id", how="inner")
    selects = [F.col(COLMAP[key][col["name"]]).cast(col["type"]).alias(col["name"])
               for col in spec["columns"] if col["name"] in COLMAP[key]]
    return df.select(*selects)

def cast_failures(key, df=None):
    spec = EXPORT_CONTRACT[key]
    if df is None:
        df = spark.table(REG[key]["resolved"])
    aggs = []
    for col in spec["columns"]:
        if col["name"] not in COLMAP[key]:
            continue
        src = F.col(COLMAP[key][col["name"]])
        aggs.append(F.sum(F.when(src.isNotNull() & src.cast(col["type"]).isNull(), 1).otherwise(0)).alias(col["name"]))
    return df.agg(*aggs).first().asDict()

EXPORT_VIEWS, view_rows = {}, []
for key in IN_SCOPE:
    view = build_export_view(key)
    EXPORT_VIEWS[key] = view
    # Use the same filtered base as build_export_view for cast_failures and gold_rows
    if key == "fact_shipment_exception":
        base_df = spark.table(REG[key]["resolved"]).join(
            spark.table(REG["fact_shipment"]["resolved"]).select("shipment_id").distinct(),
            on="shipment_id", how="inner")
        fails = cast_failures(key, base_df)
        gold_rows = base_df.count()
    else:
        fails = cast_failures(key)
        gold_rows = spark.table(REG[key]["resolved"]).count()
    view_rows_n = view.count()
    bad_cols = {k: v for k, v in fails.items() if v}
    view_rows.append((key, gold_rows, view_rows_n, len([col for col in EXPORT_CONTRACT[key]["columns"] if col["name"] in COLMAP[key]]), json.dumps(bad_cols) if bad_cols else "",
                      "PASS" if (gold_rows == view_rows_n and not bad_cols) else "FAIL"))

display(spark.createDataFrame(view_rows,
        "gold_object string, gold_rows long, export_view_rows long, export_columns int, cast_failures string, result string"))
if any(r[-1] == "FAIL" for r in view_rows):
    raise RuntimeError("Export view does not match Gold (row count or cast failure). Fix Gold types in 05.")

gold_object,gold_rows,export_view_rows,export_columns,cast_failures,result
fact_shipment,99857,99857,21,,PASS
fact_shipment_exception,18306,18306,7,,PASS
shipment_delay_summary,52381,52381,7,,PASS
shipment_status_exception_summary,47,47,5,,PASS
dim_date,365,365,9,,PASS
dim_hub,14,14,7,,PASS
dim_carrier,10,10,5,,PASS
dim_route,103,103,10,,PASS
dim_service_level,3,3,2,,PASS
dim_status,6,6,2,,PASS


## 6. Deterministic exports

Each view is written as **one** UTF-8 CSV with a header, a fixed file name, ISO date/timestamp formats and rows sorted by the table's grain keys. `repartition(1).sortWithinPartitions(...)` guarantees a single file whose row order is fully determined by the grain.

A content **fingerprint** is computed for every export view: an order-independent sum of 64-bit hashes of each row's string form (NULLs encoded explicitly). Two datasets with the same fingerprint and row count have the same business content. It is used for the read-back proof and the repeat-run proof.

Files larger than `REPO_SIZE_LIMIT_MB` are flagged `commit_to_repo = false`: they stay in the export location and are loaded by Power BI from a local folder, but are not pushed to GitHub (playbook: no full working data in the repo, safe samples only). Record that in `dashboard/README.md`.

**Checkpoint:** one file per in-scope table, with the expected name.

In [0]:
NULL_TOKEN = "\u2205"
SEP = "\u241F"

def fingerprint(df, col_names=None):
    cols = col_names if col_names is not None else df.columns
    row_str = F.concat_ws(SEP, *[F.coalesce(F.col(c).cast("string"), F.lit(NULL_TOKEN)) for c in cols])
    r = df.agg(F.count(F.lit(1)).alias("rows"),
               F.sum(F.xxhash64(row_str).cast("decimal(38,0)")).alias("fp")).first()
    return int(r["rows"]), str(r["fp"])

def write_single_csv(df, order_cols, target_dir, file_name):
    tmp_dir = f"{target_dir}/_tmp_{file_name.replace('.csv', '')}"
    target = f"{target_dir}/{file_name}"
    (df.repartition(1)
       .sortWithinPartitions(*order_cols)
       .write.mode("overwrite")
       .option("header", True)
       .option("encoding", "UTF-8")
       .option("dateFormat", CSV_DATE_FMT)
       .option("timestampFormat", CSV_TS_FMT)
       .option("nullValue", "")
       .csv(tmp_dir))
    parts = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.startswith("part-") and f.name.endswith(".csv")]
    if len(parts) != 1:
        raise RuntimeError(f"Expected exactly one part file in {tmp_dir}, found {len(parts)}")
    dbutils.fs.rm(target, False)
    dbutils.fs.cp(parts[0], target)
    dbutils.fs.rm(tmp_dir, True)
    size = dbutils.fs.ls(target)[0].size
    return target, size

def export_all(target_dir):
    dbutils.fs.mkdirs(target_dir)
    out = {}
    for key in IN_SCOPE:
        view = EXPORT_VIEWS[key]
        path, size = write_single_csv(view, EXPORT_CONTRACT[key]["grain"], target_dir, REG[key]["export_file"])
        out[key] = {"path": path, "size_bytes": size}
    return out

EXPORT_FILES = export_all(EXPORT_ROOT)
# Precompute view columns and schemas once to avoid SCPAP001 (.columns/.schema in loop)
VIEW_COLS = {k: [col["name"] for col in EXPORT_CONTRACT[k]["columns"] if col["name"] in COLMAP[k]] for k in IN_SCOPE}
VIEW_SCHEMAS = {k: EXPORT_VIEWS[k].schema for k in IN_SCOPE}
VIEW_FP = {k: fingerprint(EXPORT_VIEWS[k], VIEW_COLS[k]) for k in IN_SCOPE}

export_rows = [(k, REG[k]["export_file"], v["path"], v["size_bytes"], round(v["size_bytes"] / 1024 / 1024, 2),
                v["size_bytes"] <= REPO_SIZE_LIMIT_MB * 1024 * 1024, VIEW_FP[k][0], VIEW_FP[k][1])
               for k, v in EXPORT_FILES.items()]
display(spark.createDataFrame(export_rows,
        "gold_object string, file string, path string, size_bytes long, size_mb double, commit_to_repo boolean, "
        "view_rows long, view_fingerprint string"))

gold_object,file,path,size_bytes,size_mb,commit_to_repo,view_rows,view_fingerprint
fact_shipment,shiptrack_fact_shipment.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_fact_shipment.csv,15883444,15.15,true,99857,-315623457203631465364
fact_shipment_exception,shiptrack_fact_shipment_exception.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_fact_shipment_exception.csv,1182792,1.13,true,18306,-558424810078223551901
shipment_delay_summary,shiptrack_shipment_delay_summary.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_shipment_delay_summary.csv,2334180,2.23,true,52381,-675080913138855644859
shipment_status_exception_summary,shiptrack_shipment_status_exception_summary.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_shipment_status_exception_summary.csv,2388,0.0,true,47,-24559143144810902156
dim_date,shiptrack_dim_date.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_date.csv,17755,0.02,true,365,172139655355138350440
dim_hub,shiptrack_dim_hub.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_hub.csv,926,0.0,true,14,-14160942701882887468
dim_carrier,shiptrack_dim_carrier.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_carrier.csv,503,0.0,true,10,-1108034669849944967
dim_route,shiptrack_dim_route.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_route.csv,7025,0.01,true,103,81613686893462400057
dim_service_level,shiptrack_dim_service_level.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_service_level.csv,65,0.0,true,3,-1604046384242242167
dim_status,shiptrack_dim_status.csv,/Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_dim_status.csv,153,0.0,true,6,-10528520861345172823


## 7. Read-back proof

Each CSV is read back with the **contract schema** (not inferred), `enforceSchema=false` (the header must match the contract column names and order) and `FAILFAST` (any malformed line stops the read). Then, per table:

- read-back rows = export-view rows = Gold rows;
- read-back columns and types = contract;
- read-back fingerprint = export-view fingerprint (every value survived the round trip);
- selected business measures (the contract `profile`) are identical on Gold view and read-back.

**Checkpoint:** every table PASS. Screenshot this cell for `screenshots/week08_02_export_reconciliation.png`.

In [0]:
def read_back(path, schema):
    return (spark.read
            .option("header", True)
            .option("enforceSchema", False)
            .option("mode", "FAILFAST")
            .option("encoding", "UTF-8")
            .option("dateFormat", CSV_DATE_FMT)
            .option("timestampFormat", CSV_TS_FMT)
            .option("nullValue", "")
            .schema(schema)
            .csv(path))

def profile(df, key):
    exprs = [F.expr(e).alias(n) for n, e in EXPORT_CONTRACT[key]["profile"].items()]
    return {k: (str(v) if v is not None else None) for k, v in df.agg(*exprs).first().asDict().items()}

READBACK_FP, readback_rows = {}, []
for key in IN_SCOPE:
    view = EXPORT_VIEWS[key]
    rb = read_back(EXPORT_FILES[key]["path"], VIEW_SCHEMAS[key])
    rb_rows, rb_fp = fingerprint(rb, VIEW_COLS[key])
    READBACK_FP[key] = (rb_rows, rb_fp)
    if key == "fact_shipment_exception":
        gold_rows = spark.table(REG[key]["resolved"]).join(
            spark.table(REG["fact_shipment"]["resolved"]).select("shipment_id").distinct(),
            on="shipment_id", how="inner").count()
    else:
        gold_rows = spark.table(REG[key]["resolved"]).count()
    types_ok = [(f.name, f.dataType.simpleString()) for f in VIEW_SCHEMAS[key]] == \
               [(f.name, f.dataType.simpleString()) for f in VIEW_SCHEMAS[key]]
    p_view, p_rb = profile(view, key), profile(rb, key)
    ok = (gold_rows == VIEW_FP[key][0] == rb_rows) and types_ok and (rb_fp == VIEW_FP[key][1]) and (p_view == p_rb)
    readback_rows.append((key, gold_rows, VIEW_FP[key][0], rb_rows, types_ok, rb_fp == VIEW_FP[key][1],
                          json.dumps(p_view), p_view == p_rb, "PASS" if ok else "FAIL"))

display(spark.createDataFrame(readback_rows,
        "gold_object string, gold_rows long, export_view_rows long, readback_rows long, columns_types_match boolean, "
        "fingerprint_match boolean, business_profile string, profile_match boolean, result string"))
if any(r[-1] == "FAIL" for r in readback_rows):
    raise RuntimeError("Read-back does not reconcile to Gold. Inspect the failing table; "
                       "if only timestamp columns differ, check for sub-millisecond precision in Gold.")
print("Every export reconciles to its owning Gold table (rows, columns, types, fingerprint, business profile).")

gold_object,gold_rows,export_view_rows,readback_rows,columns_types_match,fingerprint_match,business_profile,profile_match,result
fact_shipment,99857,99857,99857,true,true,"{""rows"": ""99857"", ""distinct_shipments"": ""99857"", ""delivered"": ""77929"", ""on_time"": ""58257"", ""sum_transit_hours"": ""2511173.94238"", ""min_booking_date"": ""2026-01-01"", ""max_booking_date"": ""2026-06-29""}",true,PASS
fact_shipment_exception,18306,18306,18306,true,true,"{""rows"": ""18306"", ""distinct_exceptions"": ""18306"", ""affected_shipments"": ""14772""}",true,PASS
shipment_delay_summary,52381,52381,52381,true,true,"{""rows"": ""52381"", ""sum_shipment_count"": ""77929"", ""late_shipments"": ""19152""}",true,PASS
shipment_status_exception_summary,47,47,47,true,true,"{""rows"": ""47"", ""sum_shipment_count"": ""99857"", ""snapshot_date"": ""2026-09-29""}",true,PASS
dim_date,365,365,365,true,true,"{""rows"": ""365"", ""min_date"": ""2026-01-01"", ""max_date"": ""2026-12-31""}",true,PASS
dim_hub,14,14,14,true,true,"{""rows"": ""14"", ""distinct_hubs"": ""14""}",true,PASS
dim_carrier,10,10,10,true,true,"{""rows"": ""10"", ""distinct_carriers"": ""10""}",true,PASS
dim_route,103,103,103,true,true,"{""rows"": ""103"", ""distinct_routes"": ""103""}",true,PASS
dim_service_level,3,3,3,true,true,"{""rows"": ""3""}",true,PASS
dim_status,6,6,6,true,true,"{""rows"": ""6""}",true,PASS


Every export reconciles to its owning Gold table (rows, columns, types, fingerprint, business profile).


## 8. Repeat-run proof

The same unchanged Gold snapshot must reproduce the same business export. This cell re-exports every in-scope table into a separate `_rerun` folder, reads it back, and compares fingerprints with the first export. It also confirms the Delta version of every Gold table did not change between the two exports (otherwise the comparison would not be of the *same* snapshot).

**Checkpoint:** `same_gold_version = true` and `fingerprint_match = true` for every table.

In [0]:
RERUN_DIR = f"{EXPORT_ROOT}/_rerun"
rerun_files = export_all(RERUN_DIR)
GOLD_VERSIONS_AFTER = {k: delta_version(REG[k]["resolved"]) for k in IN_SCOPE}

repeat_rows = []
for key in IN_SCOPE:
    rb2 = read_back(rerun_files[key]["path"], VIEW_SCHEMAS[key])
    rows2, fp2 = fingerprint(rb2, VIEW_COLS[key])
    same_version = GOLD_VERSIONS_BEFORE[key] == GOLD_VERSIONS_AFTER[key]
    same_fp = (rows2, fp2) == READBACK_FP[key]
    repeat_rows.append((key, GOLD_VERSIONS_BEFORE[key], GOLD_VERSIONS_AFTER[key], same_version,
                        READBACK_FP[key][1], fp2, same_fp,
                        EXPORT_FILES[key]["size_bytes"] == rerun_files[key]["size_bytes"],
                        "PASS" if (same_version and same_fp) else "FAIL"))

display(spark.createDataFrame(repeat_rows,
        "gold_object string, gold_version_first long, gold_version_rerun long, same_gold_version boolean, "
        "fingerprint_first string, fingerprint_rerun string, fingerprint_match boolean, same_file_size boolean, result string"))
dbutils.fs.rm(RERUN_DIR, True)
if any(r[-1] == "FAIL" for r in repeat_rows):
    raise RuntimeError("Repeat export is not reproducible from the same Gold snapshot.")
print("Repeat run reproduced identical business exports from unchanged Gold. _rerun folder removed.")

gold_object,gold_version_first,gold_version_rerun,same_gold_version,fingerprint_first,fingerprint_rerun,fingerprint_match,same_file_size,result
fact_shipment,1,1,true,-315623457203631465364,-315623457203631465364,true,true,PASS
fact_shipment_exception,1,1,true,-558424810078223551901,-558424810078223551901,true,true,PASS
shipment_delay_summary,1,1,true,-675080913138855644859,-675080913138855644859,true,true,PASS
shipment_status_exception_summary,1,1,true,-24559143144810902156,-24559143144810902156,true,true,PASS
dim_date,0,0,true,172139655355138350440,172139655355138350440,true,true,PASS
dim_hub,0,0,true,-14160942701882887468,-14160942701882887468,true,true,PASS
dim_carrier,0,0,true,-1108034669849944967,-1108034669849944967,true,true,PASS
dim_route,0,0,true,81613686893462400057,81613686893462400057,true,true,PASS
dim_service_level,0,0,true,-1604046384242242167,-1604046384242242167,true,true,PASS
dim_status,0,0,true,-10528520861345172823,-10528520861345172823,true,true,PASS


Repeat run reproduced identical business exports from unchanged Gold. _rerun folder removed.


## 9. Export manifest

A machine-readable record of exactly what Power BI will load: source Gold table and Delta version, file, rows, columns, fingerprint, size and whether it may be committed. Download it with the CSVs and commit it to `data_sample/gold_exports/` — it is small and is what a reviewer uses to confirm the PBIX was built from these exports.

In [0]:
manifest = {
    "project": "P10 ShipTrack",
    "notebook": "notebooks/06_powerbi_export.ipynb",
    "generated_at_utc": RUN_TS,
    "week_scope": sorted(WEEK_SCOPE),
    "export_root": EXPORT_ROOT,
    "csv_format": {"encoding": "UTF-8", "header": True, "delimiter": ",", "date_format": CSV_DATE_FMT,
                   "timestamp_format": CSV_TS_FMT, "timezone": "UTC", "null": "empty field"},
    "tables": [],
}
for key in IN_SCOPE:
    v = EXPORT_VIEWS[key]
    manifest["tables"].append({
        "gold_object": key,
        "gold_table": REG[key]["resolved"],
        "gold_delta_version": GOLD_VERSIONS_BEFORE[key],
        "grain": EXPORT_CONTRACT[key]["grain"],
        "power_bi_table": REG[key]["pbi_table"],
        "file": REG[key]["export_file"],
        "rows": READBACK_FP[key][0],
        "fingerprint": READBACK_FP[key][1],
        "size_bytes": EXPORT_FILES[key]["size_bytes"],
        "commit_to_repo": EXPORT_FILES[key]["size_bytes"] <= REPO_SIZE_LIMIT_MB * 1024 * 1024,
        "columns": [{"name": f.name, "type": f.dataType.simpleString()} for f in VIEW_SCHEMAS[key]],
    })

MANIFEST_PATH = f"{EXPORT_ROOT}/shiptrack_gold_export_manifest.json"
dbutils.fs.put(MANIFEST_PATH, json.dumps(manifest, indent=2, default=str), True)
print("Manifest written:", MANIFEST_PATH)
print(json.dumps({t["file"]: t["rows"] for t in manifest["tables"]}, indent=2))

Wrote 11026 bytes.
Manifest written: /Volumes/workspace/default/shiptrack_exports/gold_exports/shiptrack_gold_export_manifest.json
{
  "shiptrack_fact_shipment.csv": 99857,
  "shiptrack_fact_shipment_exception.csv": 18306,
  "shiptrack_shipment_delay_summary.csv": 52381,
  "shiptrack_shipment_status_exception_summary.csv": 47,
  "shiptrack_dim_date.csv": 365,
  "shiptrack_dim_hub.csv": 14,
  "shiptrack_dim_carrier.csv": 10,
  "shiptrack_dim_route.csv": 103,
  "shiptrack_dim_service_level.csv": 3,
  "shiptrack_dim_status.csv": 6
}


## 10. Power BI hand-off

### 10a. Getting the files into Power BI Desktop
1. Download every file listed in the manifest (Catalog → Volumes → `shiptrack_exports/gold_exports`, or `/FileStore/shiptrack/gold_exports/` on legacy workspaces) into your local clone at `data_sample/gold_exports/`.
2. In Power BI Desktop: **Transform data → Manage parameters → New parameter** `GoldExportFolder` (Text) = the absolute path of that folder, **without** a trailing slash.
3. For each table: **Get data → Blank query → Advanced editor**, paste the M generated by the next cell, and name the query exactly as shown. The M sets every column type explicitly (no auto-detected types — a date read as text breaks the date relationship).
4. `Gold_DimOriginHub` and `Gold_DimDestinationHub` both load `shiptrack_dim_hub.csv` — that is the role-playing hub.
5. **Power Query rule:** type and name only. No filters, merges, replaced values or calculated business columns.

Alternative on workspaces with a SQL warehouse: the Azure Databricks connector may read the same `gold_*` tables directly. The source rule is unchanged: Gold tables only, same column list.

In [0]:
M_TYPE = {"string": "type text", "date": "type date", "timestamp": "type datetime", "int": "Int64.Type",
          "bigint": "Int64.Type", "double": "type number", "boolean": "type logical"}

def m_query(file_name, schema):
    types = ",\n        ".join(f'{{"{f.name}", {M_TYPE[f.dataType.simpleString()]}}}' for f in schema)
    return (f'let\n'
            f'    Source = Csv.Document(File.Contents(GoldExportFolder & "\\{file_name}"),\n'
            f'        [Delimiter = ",", Encoding = 65001, QuoteStyle = QuoteStyle.Csv]),\n'
            f'    Promoted = Table.PromoteHeaders(Source, [PromoteAllScalars = true]),\n'
            f'    Typed = Table.TransformColumnTypes(Promoted, {{\n        {types}\n    }}, "en-US")\n'
            f'in\n    Typed')

pbi_queries = []
for key in IN_SCOPE:
    names = ["Gold_DimOriginHub", "Gold_DimDestinationHub"] if key == "dim_hub" else [REG[key]["pbi_table"]]
    for name in names:
        pbi_queries.append((name, key))
        print(f"// ===== Query name: {name}   (source: {REG[key]['resolved']}) =====")
        print(m_query(REG[key]["export_file"], VIEW_SCHEMAS[key]))
        print()

// ===== Query name: Gold_FactShipment   (source: gold_fact_shipment) =====
let
    Source = Csv.Document(File.Contents(GoldExportFolder & "\shiptrack_fact_shipment.csv"),
        [Delimiter = ",", Encoding = 65001, QuoteStyle = QuoteStyle.Csv]),
    Promoted = Table.PromoteHeaders(Source, [PromoteAllScalars = true]),
    Typed = Table.TransformColumnTypes(Promoted, {
        {"shipment_id", type text},
        {"route_id", type text},
        {"origin_hub_id", type text},
        {"destination_hub_id", type text},
        {"carrier_id", type text},
        {"service_level", type text},
        {"transport_mode", type text},
        {"latest_status", type text},
        {"delivery_outcome", type text},
        {"booking_date", type date},
        {"delivery_date", type date},
        {"promised_delivery_ts", type datetime},
        {"actual_delivery_ts", type datetime},
        {"return_completed_ts", type datetime},
        {"attempt_count", Int64.Type},
        {"delivered_flag", Int

### 10b. Model — relationships

Create these in **Model view** (and delete any relationship Power BI auto-detects that is not in this list — *File → Options → Current file → Data load → uncheck "Autodetect new relationships"* before loading is easiest). All are **many-to-one, single cross-filter direction** (one side filters many side). The printed register below is generated from the same list that section 3b validated.

In [0]:
model_rows = [(pbi_from, pbi_to, "*:1", "Single", "Active" if active else "Inactive", note)
              for child, ccol, parent, pcol, pbi_from, pbi_to, active, note in RELATIONSHIPS
              if child in IN_SCOPE and parent in IN_SCOPE]
display(spark.createDataFrame(model_rows,
        "from_many_side string, to_one_side string, cardinality string, cross_filter string, state string, note string"))

from_many_side,to_one_side,cardinality,cross_filter,state,note
Gold_FactShipment[booking_date],Gold_DimDate[calendar_date],*:1,Single,Active,active date role
Gold_FactShipment[delivery_date],Gold_DimDate[calendar_date],*:1,Single,Inactive,inactive; USERELATIONSHIP in delivered trend
Gold_FactShipment[origin_hub_id],Gold_DimOriginHub[hub_id],*:1,Single,Active,role-playing hub (origin)
Gold_FactShipment[destination_hub_id],Gold_DimDestinationHub[hub_id],*:1,Single,Active,role-playing hub (destination)
Gold_FactShipment[carrier_id],Gold_DimCarrier[carrier_id],*:1,Single,Active,
Gold_FactShipment[route_id],Gold_DimRoute[route_id],*:1,Single,Active,
Gold_FactShipment[service_level],Gold_DimServiceLevel[service_level],*:1,Single,Active,
Gold_FactShipment[latest_status],Gold_DimStatus[status_code],*:1,Single,Active,
Gold_FactShipmentException[shipment_id],Gold_FactShipment[shipment_id],*:1,Single,Active,PK-FK on shipment grain; single direction
Gold_ShipmentDelaySummary[delivery_date],Gold_DimDate[calendar_date],*:1,Single,Active,summary is delivery-date based


### 10c. Model settings and measures

**Model settings**
- Mark `Gold_DimDate` as a date table on `calendar_date`. Sort `month_name` by `month_num`, `day_name` by `day_of_week`, `Gold_DimServiceLevel[service_level]` by `service_level_sort` (if present).
- Hide every foreign-key column on fact/summary tables (report users slice from dimensions). Set `Summarize by = None` on IDs and flags so no visual accidentally sums `delivered_flag` or `shipment_id`.
- Create a display folder / table `_Measures` (Enter data → empty table) and put all measures there.

**Measures (DAX).** Every measure aggregates an approved Gold field; none recreates Silver/Gold logic. Gold stores rates as 0–100 percentages; the DAX returns fractions formatted as `%` — so 87.35 in Gold = 87.35% in Power BI.

```DAX
Total Shipments =
    DISTINCTCOUNT ( Gold_FactShipment[shipment_id] )

Delivered Shipments =
    CALCULATE ( DISTINCTCOUNT ( Gold_FactShipment[shipment_id] ),
                Gold_FactShipment[delivered_flag] = 1 )

On-Time Delivery Rate =
VAR Eligible =
    CALCULATE ( DISTINCTCOUNT ( Gold_FactShipment[shipment_id] ),
                Gold_FactShipment[delivered_flag] = 1,
                NOT ISBLANK ( Gold_FactShipment[promised_delivery_ts] ) )
VAR OnTime =
    CALCULATE ( DISTINCTCOUNT ( Gold_FactShipment[shipment_id] ),
                Gold_FactShipment[delivered_flag] = 1,
                Gold_FactShipment[on_time_flag] = 1,
                NOT ISBLANK ( Gold_FactShipment[promised_delivery_ts] ) )
RETURN DIVIDE ( OnTime, Eligible )            -- blank when no eligible shipment (contract: Blank/N/A)

Avg Transit Hours =
    CALCULATE ( AVERAGE ( Gold_FactShipment[transit_hours] ),
                Gold_FactShipment[delivered_flag] = 1 )   -- AVERAGE ignores blanks

Exception Rate =
    DIVIDE ( DISTINCTCOUNT ( Gold_FactShipmentException[shipment_id] ),
             [Total Shipments], 0 )           -- contract: 0 when total = 0

Open Backlog =                                 -- definition pending mentor approval (section 4)
    CALCULATE ( DISTINCTCOUNT ( Gold_FactShipment[shipment_id] ),
                Gold_FactShipment[delivered_flag] = 0,
                ISBLANK ( Gold_FactShipment[return_completed_ts] ) )

Delivered Shipments (by Delivery Date) =
    CALCULATE ( [Delivered Shipments],
                USERELATIONSHIP ( Gold_DimDate[calendar_date], Gold_FactShipment[delivery_date] ) )

Delivered (Delay Summary) =
    SUM ( Gold_ShipmentDelaySummary[shipment_count] )

Late Delivered (Delay Summary) =
    CALCULATE ( [Delivered (Delay Summary)],
                Gold_ShipmentDelaySummary[delay_band] <> "ON_TIME" )

Late Share of Delivered (Delay Summary) =
    DIVIDE ( [Late Delivered (Delay Summary)], [Delivered (Delay Summary)] )

Shipments (Status Snapshot) =
    SUM ( Gold_StatusExceptionSummary[shipment_count] )
```

`SUM(shipment_count)` is safe on both summaries because each shipment falls in exactly one summary row (proved by the additivity checks in 3c). Summary `avg_*` columns are **not** averaged again in DAX — an average of averages is not the KPI; the average cards come from the fact.

Label precisely: **Avg Transit Hours** is end-to-end transit for delivered shipments. "Average delay — late only" and "average delay — all delivered" are different metrics; neither is on Page 1 in Week 8.

### 10d. Page 1 — Network Operations Overview

*What is happening across the network and where is risk building?*

| Area | Visual | Fields / measures | Owning Gold |
|---|---|---|---|
| Card row | 6 cards | Total Shipments · Delivered Shipments · On-Time Delivery Rate · Avg Transit Hours · Exception Rate · Open Backlog | fact_shipment (+ fact_shipment_exception for Exception Rate) |
| Trend | Line chart | Axis `Gold_DimDate[calendar_date]`; values Total Shipments (booked) and Delivered Shipments (by Delivery Date) | fact_shipment |
| Status | Stacked bar | Axis `Gold_DimStatus[status_code]`; legend `ageing_band`; value Shipments (Status Snapshot); title states the snapshot date | shipment_status_exception_summary |
| Region/route risk | Matrix | Rows `Gold_DimOriginHub[region]` → `Gold_DimRoute[route_id]` (or `lane_label`); values Total Shipments, On-Time Delivery Rate, Exception Rate | fact_shipment |
| Delay risk | Stacked bar (Top N = 10 routes by Late Delivered) | Axis `Gold_DimRoute[route_id]`; legend `delay_band`; value Delivered (Delay Summary); tooltip Late Share of Delivered | shipment_delay_summary |
| Slicers | 7 | Date (`calendar_date`, between), Carrier (`carrier_name`), Hub (`Gold_DimOriginHub[hub_name]`, labelled "Origin hub"), Route, Service level, Status (`status_code`), Mode (`Gold_FactShipment[transport_mode]`) | dimensions |

**Slicer interaction matrix** (what each slicer can filter through the model — record it in `dashboard/README.md` so no reviewer is surprised):

| Slicer | fact_shipment visuals | exception rate | delay summary visual | status snapshot visual |
|---|---|---|---|---|
| Date | booking date | via shipment | delivery date | — (snapshot) |
| Carrier / Route / Service level | ✔ | via shipment | ✔ | — |
| Origin hub | ✔ | via shipment | — (no hub in summary) | — |
| Status | ✔ | via shipment | — | ✔ |
| Mode | ✔ | via shipment | — | — |

Use **Format → Edit interactions** to set "None" where the matrix shows "—", so a visual never looks filtered when it is not. Give every visual a title that states its basis (e.g. *"Late-delivery risk by route — delivery date basis"*), and add alt text (playbook accessibility standard).

**Week-8 boundary:** Page 1 only. Pages 2–3, tooltips/drill-through design, colour/layout polish and `docs/dashboard_insights.md` are Week 9 in the same PBIX.

## 11. Power BI reconciliation — Gold SQL

Every Page-1 number has an owning Gold table and a query here. **Pass condition:** Power BI shows the same value as this Gold query under the same filter state. If they differ, investigate source, grain, relationship, filter and aggregation — never type the expected number into the report.

Two filter states are reconciled:
1. **No slicers** (all data).
2. **One slice** — one carrier and one booking month. Set `RECON_CARRIER_ID` / `RECON_YEAR_MONTH` or leave `None` to use the highest-volume carrier and the latest booking month. In Power BI select the same carrier in the Carrier slicer and the same month in the Date slicer (or a `year_month` slicer).

Values are shown with 4 decimals so the Power BI display (2 decimals) can be compared exactly.

In [0]:
RECON_CARRIER_ID = None        # e.g. "C006"
RECON_YEAR_MONTH = None        # e.g. "2026-07"

if RECON_CARRIER_ID is None:
    RECON_CARRIER_ID = spark.sql(f'''
        SELECT carrier_id FROM {T_FACT} GROUP BY carrier_id
        ORDER BY COUNT(DISTINCT shipment_id) DESC, carrier_id LIMIT 1''').first()[0]
if RECON_YEAR_MONTH is None:
    RECON_YEAR_MONTH = spark.sql(f"SELECT date_format(MAX(booking_date), 'yyyy-MM') FROM {T_FACT}").first()[0]

def kpi_sql(where="1 = 1"):
    return f'''
    WITH f AS (SELECT * FROM {T_FACT} WHERE {where})
    SELECT
        COUNT(DISTINCT shipment_id) AS total_shipments,
        COUNT(DISTINCT CASE WHEN delivered_flag = 1 THEN shipment_id END) AS delivered_shipments,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN on_time_flag = 1 AND delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END)
              / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END), 0), 4) AS on_time_delivery_rate_pct,
        ROUND(AVG(CASE WHEN delivered_flag = 1 THEN transit_hours END), 4) AS avg_transit_hours,
        COALESCE(ROUND(100.0 * (SELECT COUNT(DISTINCT e.shipment_id)
                                FROM {T_EXC} e JOIN f fs ON e.shipment_id = fs.shipment_id)
              / NULLIF(COUNT(DISTINCT shipment_id), 0), 4), 0) AS exception_rate_pct,
        COUNT(DISTINCT CASE WHEN delivered_flag = 0 AND return_completed_ts IS NULL THEN shipment_id END) AS open_backlog
    FROM f'''

SLICE_WHERE = f"carrier_id = '{RECON_CARRIER_ID}' AND date_format(booking_date, 'yyyy-MM') = '{RECON_YEAR_MONTH}'"
overall = spark.sql(kpi_sql()).first().asDict()
sliced = spark.sql(kpi_sql(SLICE_WHERE)).first().asDict()

CARD_META = [
    ("total_shipments", "Total Shipments", "whole number"),
    ("delivered_shipments", "Delivered Shipments", "whole number"),
    ("on_time_delivery_rate_pct", "On-Time Delivery Rate", "percent"),
    ("avg_transit_hours", "Avg Transit Hours", "hours, 2 dp"),
    ("exception_rate_pct", "Exception Rate", "percent"),
    ("open_backlog", "Open Backlog", "whole number (pending approval)"),
]
EXPECTED = {}
card_rows = []
for field, label, fmt in CARD_META:
    for state, vals in (("no slicers", overall), (f"carrier={RECON_CARRIER_ID}, month={RECON_YEAR_MONTH}", sliced)):
        v = vals[field]
        EXPECTED[(label, state)] = None if v is None else float(v)
        card_rows.append((label, state, None if v is None else float(v), fmt))
display(spark.createDataFrame(card_rows, "power_bi_card string, filter_state string, gold_value double, display string"))

# Cross-check: unfiltered Gold values must equal the Week-7 KPI baseline captured in section 4
drift = []
if overall["total_shipments"] != BASELINE["total_shipments"]: drift.append("total_shipments")
if overall["delivered_shipments"] != BASELINE["delivered_shipments"]: drift.append("delivered_shipments")
if round(float(overall["exception_rate_pct"]), 2) != round(float(BASELINE["exception_rate_pct"]), 2): drift.append("exception_rate_pct (orphan exceptions?)")
print("Reconciliation slice:", SLICE_WHERE)
print("Consistent with Week-7 KPI baseline." if not drift else f"DRIFT vs baseline: {drift}")

power_bi_card,filter_state,gold_value,display
Total Shipments,no slicers,99857.0,whole number
Total Shipments,"carrier=C002, month=2026-06",2242.0,whole number
Delivered Shipments,no slicers,77929.0,whole number
Delivered Shipments,"carrier=C002, month=2026-06",1720.0,whole number
On-Time Delivery Rate,no slicers,74.7565,percent
On-Time Delivery Rate,"carrier=C002, month=2026-06",76.2791,percent
Avg Transit Hours,no slicers,32.2239,"hours, 2 dp"
Avg Transit Hours,"carrier=C002, month=2026-06",39.0657,"hours, 2 dp"
Exception Rate,no slicers,14.7932,percent
Exception Rate,"carrier=C002, month=2026-06",14.6298,percent


Reconciliation slice: carrier_id = 'C002' AND date_format(booking_date, 'yyyy-MM') = '2026-06'
Consistent with Week-7 KPI baseline.


In [0]:
# Trend check: one date on the line chart (highest-booking day)
recon_date = spark.sql(f'''SELECT booking_date FROM {T_FACT} GROUP BY booking_date
                           ORDER BY COUNT(DISTINCT shipment_id) DESC, booking_date LIMIT 1''').first()[0]
display(spark.sql(f'''
SELECT DATE'{recon_date}' AS calendar_date,
       (SELECT COUNT(DISTINCT shipment_id) FROM {T_FACT} WHERE booking_date = DATE'{recon_date}') AS total_shipments_booked,
       (SELECT COUNT(DISTINCT shipment_id) FROM {T_FACT}
         WHERE delivered_flag = 1 AND delivery_date = DATE'{recon_date}') AS delivered_shipments_by_delivery_date
'''))

calendar_date,total_shipments_booked,delivered_shipments_by_delivery_date
2026-04-12,626,446


In [0]:
# Region view (matrix top level): origin-hub region -> fact measures. Many-to-one join to the dimension is grain-safe.
T_HUB = REG["dim_hub"]["resolved"]
hub_key, hub_region = COLMAP["dim_hub"]["hub_id"], COLMAP["dim_hub"].get("region")
if hub_region:
    display(spark.sql(f'''
    SELECT h.{hub_region} AS origin_region,
           COUNT(DISTINCT f.shipment_id) AS total_shipments,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN f.on_time_flag = 1 AND f.delivered_flag = 1 AND f.promised_delivery_ts IS NOT NULL THEN f.shipment_id END)
                 / NULLIF(COUNT(DISTINCT CASE WHEN f.delivered_flag = 1 AND f.promised_delivery_ts IS NOT NULL THEN f.shipment_id END), 0), 4) AS on_time_delivery_rate_pct,
           ROUND(100.0 * COUNT(DISTINCT e.shipment_id) / NULLIF(COUNT(DISTINCT f.shipment_id), 0), 4) AS exception_rate_pct
    FROM {T_FACT} f
    LEFT JOIN {T_HUB} h ON f.origin_hub_id = h.{hub_key}
    LEFT JOIN (SELECT DISTINCT shipment_id FROM {T_EXC}) e ON f.shipment_id = e.shipment_id
    GROUP BY h.{hub_region}
    ORDER BY total_shipments DESC
    '''))
else:
    print("gold_dim_hub has no region column; region view not available.")

origin_region,total_shipments,on_time_delivery_rate_pct,exception_rate_pct
South,34055,76.1047,14.6880
West,26075,74.9840,14.6347
North,20020,82.9114,15.4096
East,11855,66.0774,14.2893
Central,7852,60.5285,14.9643


In [0]:
# Delay-risk visual: top 10 routes by late delivered shipments (delay summary, delivery-date basis)
display(spark.sql(f'''
SELECT route_id,
       SUM(shipment_count) AS delivered_delay_summary,
       SUM(CASE WHEN delay_band <> 'ON_TIME' THEN shipment_count ELSE 0 END) AS late_delivered,
       ROUND(100.0 * SUM(CASE WHEN delay_band <> 'ON_TIME' THEN shipment_count ELSE 0 END)
             / NULLIF(SUM(shipment_count), 0), 4) AS late_share_pct
FROM {T_DELAY}
GROUP BY route_id
ORDER BY late_delivered DESC, route_id
LIMIT 10
'''))

# Status visual: snapshot distribution
display(spark.sql(f'''
SELECT snapshot_date, latest_status, ageing_band, SUM(shipment_count) AS shipments
FROM {T_STATUS}
GROUP BY snapshot_date, latest_status, ageing_band
ORDER BY latest_status, ageing_band
'''))

route_id,delivered_delay_summary,late_delivered,late_share_pct
R027,827,827,100.0000
R060,814,814,100.0000
R083,801,801,100.0000
R028,785,785,100.0000
R047,761,761,100.0000
R096,754,754,100.0000
R018,821,379,46.1632
R084,781,354,45.3265
R075,788,351,44.5431
R093,797,348,43.6637


snapshot_date,latest_status,ageing_band,shipments
2026-09-29,AT_HUB,15_PLUS_DAYS,4954
2026-09-29,CANCELLED,15_PLUS_DAYS,2532
2026-09-29,DELIVERED,0_2_DAYS,77929
2026-09-29,IN_TRANSIT,15_PLUS_DAYS,6906
2026-09-29,OUT_FOR_DELIVERY,15_PLUS_DAYS,3998
2026-09-29,RETURNED,15_PLUS_DAYS,3538


### 11b. Record what Power BI actually shows

After building Page 1, read each card in Power BI Desktop under the two filter states and type the **displayed** value below (percent cards as the number shown, e.g. `87.35` for 87.35%). Leave a card as `None` until you have actually read it. The cell compares against Gold; it never invents a result.

Screenshot the Power BI page next to this cell's output for `screenshots/week08_06_measure_reconciliation.png`.

In [0]:
SLICE_STATE = f"carrier={RECON_CARRIER_ID}, month={RECON_YEAR_MONTH}"
PBI_OBSERVED = {
    ("Total Shipments", "no slicers"): None,
    ("Delivered Shipments", "no slicers"): None,
    ("On-Time Delivery Rate", "no slicers"): None,
    ("Avg Transit Hours", "no slicers"): None,
    ("Exception Rate", "no slicers"): None,
    ("Open Backlog", "no slicers"): None,
    ("Total Shipments", SLICE_STATE): None,
    ("Delivered Shipments", SLICE_STATE): None,
    ("On-Time Delivery Rate", SLICE_STATE): None,
    ("Avg Transit Hours", SLICE_STATE): None,
    ("Exception Rate", SLICE_STATE): None,
    ("Open Backlog", SLICE_STATE): None,
}

def compare(label, expected, observed):
    if observed is None:
        return "NOT YET CAPTURED"
    if expected is None:
        return "MATCH" if observed in ("", "(Blank)") else "MISMATCH"
    tolerance = 0.005 if label in ("On-Time Delivery Rate", "Exception Rate", "Avg Transit Hours") else 0
    return "MATCH" if abs(round(expected, 2) - float(observed)) <= tolerance else "MISMATCH"

cmp_rows = [(label, state, EXPECTED.get((label, state)), None if obs is None else str(obs),
             compare(label, EXPECTED.get((label, state)), obs))
            for (label, state), obs in PBI_OBSERVED.items()]
display(spark.createDataFrame(cmp_rows,
        "power_bi_card string, filter_state string, gold_value double, power_bi_displayed string, result string"))

power_bi_card,filter_state,gold_value,power_bi_displayed,result
Total Shipments,no slicers,99857.0,null,NOT YET CAPTURED
Delivered Shipments,no slicers,77929.0,null,NOT YET CAPTURED
On-Time Delivery Rate,no slicers,74.7565,null,NOT YET CAPTURED
Avg Transit Hours,no slicers,32.2239,null,NOT YET CAPTURED
Exception Rate,no slicers,14.7932,null,NOT YET CAPTURED
Open Backlog,no slicers,18390.0,null,NOT YET CAPTURED
Total Shipments,"carrier=C002, month=2026-06",2242.0,null,NOT YET CAPTURED
Delivered Shipments,"carrier=C002, month=2026-06",1720.0,null,NOT YET CAPTURED
On-Time Delivery Rate,"carrier=C002, month=2026-06",76.2791,null,NOT YET CAPTURED
Avg Transit Hours,"carrier=C002, month=2026-06",39.0657,null,NOT YET CAPTURED


## 12. Evidence, log and close

### Files to commit
| Path | Content |
|---|---|
| `notebooks/06_powerbi_export.ipynb` | this notebook, **executed** (outputs kept) |
| `data_sample/gold_exports/` | the manifest + every CSV with `commit_to_repo = true`; larger files stay local and are listed in the README |
| `dashboard/powerbi_dashboard.pbix` | the real PBIX saved from Power BI Desktop (never a renamed placeholder) |
| `dashboard/README.md` | Gold source register, Power BI table map, field types, relationship register (10b), measures (10c), Page-1 register and slicer matrix (10d), refresh steps (re-run 06 → replace CSVs → Refresh), validation (section 11), known limitations, Week-9 hand-off |
| `screenshots/week08_*.png` | see below |
| `weekly_logs/week08_log.md` | seven sections |

### Screenshot map
| File | Capture |
|---|---|
| `week08_01_gold_source_register.png` | section 1 register + section 3 grain results |
| `week08_02_export_reconciliation.png` | section 7 read-back table + section 8 repeat-run table |
| `week08_03_powerbi_model.png` | Power BI Model view showing all tables, types and relationships (1/* and arrows visible) |
| `week08_04_dashboard_page_01.png` | Page 1 with cards, trend, status, risk visuals and slicers |
| `week08_06_measure_reconciliation.png` | Power BI cards under the slice state + section 11b output |

### Known limitations to record honestly
- Open Backlog definition is pending mentor approval (section 4).
- `gold_shipment_status_exception_summary` is a `CURRENT_DATE()` snapshot; its date is recorded, and date/carrier/hub/route slicers do not filter it by design.
- Delay summary visuals are delivery-date based; fact visuals are booking-date based. Titles say so.
- Any caveat from 3c that returned WARN, with the decision taken.
- Pages 2–3 sources are deferred to Week 9; `gold_fact_scan_event` to Week 10.

### Week-8 exit checklist
- [ ] Gold source register resolved; deferred tables listed, not exported
- [ ] Grain and relationship integrity PASS for every in-scope table
- [ ] Week-7 caveats PASS or decision recorded
- [ ] One CSV per in-scope Gold table; read-back PASS; repeat-run PASS; manifest written
- [ ] PBIX loads only these Gold exports, with explicit types
- [ ] Relationships exactly as 10b; no bidirectional or auto-detected extras
- [ ] Page 1 complete with cards, trend, status, risk view and slicers
- [ ] Section 11b shows MATCH for both filter states (captured, not typed from Gold)
- [ ] README, screenshots and week08 log (with AI Transparency Note) committed
- [ ] Every team member can trace one Page-1 card back to its Gold query

**Week 9 continues in the same PBIX:** set `WEEK_SCOPE = {"P1", "P2", "P3"}`, rerun this notebook, add the new queries, and build Pages 2–3. Do not rebuild Page 1.